In [ ]:
import re
import pandas as pd

In [ ]:
patients_df = pd.read_csv("patients.csv")
print(patients_df.head())

In [ ]:
treatments_df = pd.read_csv("treatments.csv")
treatments_cut_df = pd.read_csv("treatments_cut.csv")
adverse_reaction_df = pd.read_csv("adverse_reactions.csv")

In [ ]:
patients_df.info()

In [ ]:
patients_df = patients_df.fillna("No data")

In [ ]:
treatments_df = pd.concat([treatments_df, treatments_cut_df])

In [ ]:
treatments_df["hba1c_change"] = treatments_df["hba1c_start"] - treatments_df["hba1c_end"]
print(treatments_df.sample(5))

In [ ]:
def separate_contact(contact):
    if pd.isna(contact):
        return pd.Series({"mobile": None, "email": None})
    email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    mobile_pattern = r"(\+?1?\s*\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}|\d{3}[-.\s]\d{3}[-.\s]\d{4})"
    email = re.search(email_pattern, contact)
    remaining = re.sub(email_pattern, "", contact) if email else contact
    email_match = email.group() if email else None
    mobile_match = re.search(mobile_pattern, remaining.replace(" ", ""))
    mobile = mobile_match.group() if mobile_match else None
    return pd.Series({"mobile": mobile, "email": email_match})

patients_df[["mobile", "email"]] = patients_df["contact"].apply(separate_contact)
print(patients_df.sample(5))

In [ ]:
patients_df = patients_df.drop(columns=["contact"])
patients_df.info()
treatments_df.info()

In [ ]:
def separate_dosage(value):
    if pd.isna(value) or value == "-":
        return pd.Series({"start": None, "end": None})
    parts = str(value).split(" - ")
    return pd.Series({
        "start": parts[0].replace("u", "") if len(parts) > 0 else None,
        "end": parts[1].replace("u", "") if len(parts) > 1 else None,
    })

treatments_df[["auralin_start", "auralin_end"]] = treatments_df["auralin"].apply(separate_dosage)
treatments_df[["novodra_start", "novodra_end"]] = treatments_df["novodra"].apply(separate_dosage)
treatments_df.sample(5)

In [ ]:
treatments_df = treatments_df.drop(columns=["auralin", "novodra"])
treatments_df.info()

In [ ]:
treatments_df.sample(5)

In [ ]:
treatments_df = pd.read_csv("treatments.csv")
treatments_df = pd.concat([treatments_df, treatments_cut_df])
treatments_df["hba1c_change"] = treatments_df["hba1c_start"] - treatments_df["hba1c_end"]
treatments_df.info()

In [ ]:
treatments_df = treatments_df.melt(
    id_vars=["given_name", "surname", "hba1c_start", "hba1c_end", "hba1c_change"],
    var_name="type",
    value_name="dosage",
)
treatments_df.sample(2)

In [ ]:
treatments_df = treatments_df[treatments_df["dosage"] != "-"]
treatments_df.sample(4)

In [ ]:
treatments_df.info()
treatments_df["dosage_start"] = treatments_df["dosage"].str.split("-").str.get(0)
treatments_df["dosage_end"] = treatments_df["dosage"].str.split("-").str.get(1)

In [ ]:
treatments_df = treatments_df.drop(columns=["dosage"])
treatments_df.sample(3)

In [ ]:
treatments_df["dosage_start"] = treatments_df["dosage_start"].str.replace("u", "")
treatments_df["dosage_end"] = treatments_df["dosage_end"].str.replace("u", "")
treatments_df.head(2)
treatments_df.info()

In [ ]:
treatments_df = treatments_df.merge(adverse_reaction_df, how="left", on=["given_name", "surname"])
treatments_df.sample(4)

In [ ]:
treatments_df["dosage_start"] = treatments_df["dosage_start"].astype("int")
treatments_df["dosage_end"] = treatments_df["dosage_end"].astype("int")
treatments_df["given_name"] = treatments_df["given_name"].str.capitalize()
treatments_df["surname"] = treatments_df["surname"].str.capitalize()
treatments_df

## Data Quality Issues Identified:
- Typo in patient_id 9 (accuracy)
- Short forms in state (consistency)
- Zip code should be 5 digits - some contain 4 digits (validity)
- Missing address, zip, state, city, country, contact (completeness)
- Duplicate patients named 'John Doe' (accuracy)
- Weight = 48 is incorrect (accuracy)
- Height = 4.411297 is incorrect (accuracy)
- Birthday should be datetime type (validity)
- Zip code should be integer type (validity)
- Incorrect data type for sex

In [ ]:
patients_df